In [ ]:
!git clone https://github.com/jwkirchenbauer/lm-watermarking
%cd lm-watermarking
!pip install -r requirements.txt


In [ ]:
import torch
print("Torch version:", torch.__version__)
print("GPU available:", torch.cuda.is_available())

In [ ]:
import torch
print("GPU available:", torch.cuda.is_available())

In [ ]:
import torch
print(torch.__version__)
print(torch.cuda.get_device_name(0))

In [ ]:
!ls

!git clone https://github.com/jwkirchenbauer/lm-watermarking
%cd lm-watermarking
!pip install -r requirements.txt


In [ ]:
!ls

In [ ]:
!python demo_watermark.py

In [ ]:
!python demo_watermark.py --model_name_or_path gpt2

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

# These class names exist in this repo
from watermark_processor import WatermarkLogitsProcessor, WatermarkDetector

device = "cuda" if torch.cuda.is_available() else "cpu"
model_name = "gpt2"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name).to(device)
model.eval()

prompt = "Explain in simple terms what an LLM watermark is and why it matters."

# Watermark params (same idea as demo)
gamma = 0.25
delta = 2.0
seeding_scheme = "simple_1"

# Build watermark objects
wm_processor = WatermarkLogitsProcessor(vocab=list(tokenizer.get_vocab().values()),
                                       gamma=gamma, delta=delta,
                                       seeding_scheme=seeding_scheme,
                                       select_green_tokens=True)

detector = WatermarkDetector(vocab=list(tokenizer.get_vocab().values()),
                             gamma=gamma,
                             seeding_scheme=seeding_scheme,
                             device=device,
                             tokenizer=tokenizer,
                             z_threshold=4.0,
                             select_green_tokens=True)

def generate_text(use_watermark: bool):
    inputs = tokenizer(prompt, return_tensors="pt").to(device)
    logits_processors = [wm_processor] if use_watermark else None

    out = model.generate(
        **inputs,
        max_new_tokens=180,
        do_sample=True,
        temperature=0.7,
        logits_processor=logits_processors
    )
    return tokenizer.decode(out[0], skip_special_tokens=True)

normal = generate_text(use_watermark=False)
watermarked = generate_text(use_watermark=True)

print("=== NORMAL ===")
print(normal[-800:])  # show last part
print("\nDetection (normal):", detector.detect(normal))

print("\n=== WATERMARKED ===")
print(watermarked[-800:])
print("\nDetection (watermarked):", detector.detect(watermarked))

In [ ]:
attack_prompt = "Rewrite the following text using different wording but keep the same meaning:\n\n" + watermarked

inputs = tokenizer(attack_prompt, return_tensors="pt").to(device)
out = model.generate(
    **inputs,
    max_new_tokens=220,
    do_sample=True,
    temperature=0.9
)
paraphrased = tokenizer.decode(out[0], skip_special_tokens=True)

print("=== PARAPHRASED (ATTACK) ===")
print(paraphrased[-800:])
print("\nDetection (paraphrased):", detector.detect(paraphrased))

In [ ]:
!ls -la